# Predictive & Optimization Analytics: Comprehensive Gold Price Forecasting

**Student:** James Mithiga
**Admission No:** 58200
**Date:** March 2026

## Project Overview

This analysis implements and compares six forecasting architectures to predict Gold (GC=F) closing prices using a five-year rolling dataset through February 28, 2026. The study evaluates econometric, machine learning, and deep learning approaches using identical train-test splits to ensure fair comparison and eliminate data leakage.

**Models Evaluated:**
- ARIMA and SARIMA (econometric baselines)
- Linear Regression and Random Forest (machine learning)
- Facebook Prophet (Bayesian structural time-series)
- LSTM Neural Networks (deep learning)

## Evaluation Framework

All models are evaluated on a chronologically-separated test set comprising the final twenty percent of the dataset. Performance is assessed via RMSE, MAE, MAPE, and directional accuracy metrics presented in comparative tables. See summary section for complete performance results.

### Task 1: Data Preparation

#### 1.1 Data Acquisition and Validation

Data for Gold futures (GC=F) is retrieved from Yahoo Finance spanning five years ending February 28, 2026. The ingestion process standardizes OHLCV format, converts indices to datetime objects, applies forward-fill to handle missing values, and removes remaining gaps. This ensures consistent, clean data for all downstream modeling.

In [1]:
import pmdarima as pm
from statsmodels.tsa.stattools import adfuller
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import logging
import cmdstanpy
import warnings
import itertools
import torch.nn.init as init
from torch.utils.data import TensorDataset, DataLoader
import time
import os
import sys
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from statsmodels.tsa.stattools import acf, pacf
import pandas_ta as ta
import talib
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
plt.style.use('ggplot')

# Initialize an empty list to accumulate performance metrics from each model.
results_list = []

c:\Users\James Mithiga\Dropbox\Strathmore MSc. DSA\Units\Module 5\Streamlit\Deployment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'talib'

In [ ]:
# Define a function that persists trained models to disk in the models directory using joblib serialization.
import joblib
def save_model(model, ticker, model_name):
    if not os.path.exists('models'):
        os.makedirs('models')
    filename = f'models/{ticker}_{model_name}.pkl'
    joblib.dump(model, filename)


In [ ]:
def calculate_metrics(y_true, y_pred, model_name, ticker):
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    y_true, y_pred = np.array(y_true).ravel(), np.array(y_pred).ravel()
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    nonzero_idx = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero_idx] - y_pred[nonzero_idx]) / y_true[nonzero_idx])) * 100 if np.any(nonzero_idx) else np.nan
    r2 = r2_score(y_true, y_pred)
    diff_true = np.diff(y_true)
    diff_pred = np.diff(y_pred)
    da = np.mean(np.sign(diff_true) == np.sign(diff_pred)) * 100 if len(diff_true) > 0 else np.nan
    return {'Ticker': ticker, 'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2 Score': r2, 'Directional Accuracy (%)': da}

#### 1.2 Visualization and Integrity Checks

Interactive visualizations reveal price trajectories with volatility regime shifts. Data quality inspection confirms the absence of significant gaps, validating the integrity of the dataset.

In [ ]:
ticker = 'GC=F'
cutoff_date = '2026-02-28'

# Retrieve the dataset from Yahoo Finance for the specified ticker spanning five years.
raw = yf.download(ticker, period='5y')

# Extract the close price series and ensure consistent column naming.
if isinstance(raw.columns, pd.MultiIndex):
    df = raw['Close'][[ticker]]
    df.columns = ['Close']
else:
    df = raw[['Close']]

# Restrict the time series to observations on or before the cutoff date.
df = df.loc[df.index <= cutoff_date]

# Fill forward any missing values and remove remaining gaps to ensure data integrity.
missing = df.isnull().sum()
print(f'Missing values before handling: {missing.Close}')
df = df.ffill().dropna()

print(f'Cutoff Applied: {cutoff_date}')
print(f'Total trading days up to cutoff: {len(df)}')

df.tail()


[*********************100%***********************]  1 of 1 completed

Missing values before handling: 0
Cutoff Applied: 2026-02-28
Total trading days up to cutoff: 1248


,Close
Date,
2026-02-23,5204.700195
2026-02-24,5155.799805
2026-02-25,5206.399902
2026-02-26,5176.500000
2026-02-27,5230.500000


#### 1.3 Missing Value Assessment

In [ ]:
missing_values = df.isnull().sum()
missing_values

Close    0
dtype: int64

In [ ]:
# Create a new plotly figure object for interactive visualization.
fig = go.Figure()

# Add the closing price as a continuous line trace to the figure.
fig.add_trace(go.Scatter(
x=df.index,
y=df['Close'],
mode='lines',
name=f"{ticker} Price",
line=dict(color='#00D1FF', width=2),
hovertemplate="<b>Date:</b> %{x}<br><b>Price:</b> $%{y:.2f}<extra></extra>"
))

# Configure the figure layout with title, axes labels, and styling options.
fig.update_layout(
title={
'text': f"{ticker} 5-Year Historical Price (USD)",
'y':0.95,
'x':0.5,
'xanchor': 'center',
'yanchor': 'top',
'font': dict(size=20)
},
xaxis_title="Year",
yaxis_title="Stock Price (USD)",
template='plotly_dark',
hovermode='x unified',
)

# Display the interactive visualization.
fig.show()


Visual inspection confirms continuity in price curves, indicating successful data ingestion. Stationarity testing proceeds on the clean dataset.

### Task 2: Stationarity Analysis and Parameter Selection

#### 2.1 Augmented Dickey-Fuller Testing

ADF testing reveals non-stationarity in raw gold prices, indicating a significant trend. First-order differencing (d=1) removes the trend, while seasonal differencing (D=1, s=21) accounts for the 21-day trading cycle. Post-transformation testing confirms stationarity, enabling ARIMA framework application.

#### 2.2 Autocorrelation Diagnostics

ACF and PACF plots guide parameter selection for ARIMA modeling. Significant lags in PACF inform the autoregressive order (p), while ACF cutoffs suggest moving-average order (q). These visual diagnostics complement the `auto_arima` algorithm for parameter optimization.

In [ ]:
def check_stat(df):
    if df.empty:
        print("The DataFrame is empty!")
        return

    for col in df.columns:
        try:
            # Remove null values from the series because the Augmented Dickey-Fuller test requires complete data.
            series = df[col].dropna()
            result = adfuller(series)
            p_val = result[1]
            status = 'Stationary' if p_val <= 0.05 else 'Non-Stationary'
            print(f"{col:15} | p-value: {p_val:.4f} | {status}")
        except Exception as e:
            print(f"Could not process {col}: {e}")


#### 2.3 Stationarity Verification

Re-testing post-transformation confirms successful trend removal, enabling ARIMA to model stochastic components.

In [ ]:
check_stat(df[['Close']].diff().dropna())

Close           | p-value: 0.0000 | Stationary


#### 2.4 Lag Relationship Visualization

Plots display autocorrelation patterns across lags, informing parameter selection.

In [ ]:
# Establish the time series data for analysis by computing first-order differences to remove the trend component.
diff_series = df['Close'].diff().dropna()
n_obs = len(diff_series)

# Compute autocorrelation and partial autocorrelation values across 40 lags.
lags = 40
acf_vals = acf(diff_series, nlags=lags)
pacf_vals = pacf(diff_series, nlags=lags, method='ywm')

# Compute the 95 percent confidence interval bounds for statistical significance testing.
ci_value = 1.96 / np.sqrt(n_obs)

# Instantiate plotly subplots for side-by-side ACF and PACF visualization.
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Autocorrelation (ACF)', 'Partial Autocorrelation (PACF)'),
    horizontal_spacing=0.1
)

def add_corr_traces(fig, vals, row, col):
    # Draw vertical lines from zero to each correlation value to represent the magnitude of correlation.
    for i, val in enumerate(vals):
        fig.add_trace(
            go.Scatter(
                x=[i, i], y=[0, val],
                mode='lines',
                line=dict(color='#00D1FF', width=1),
                hoverinfo='skip',
                showlegend=False
            ),
            row=row, col=col
        )

    # Overlay circular markers at the end of each vertical line to highlight individual values.
    fig.add_trace(
        go.Scatter(
            x=np.arange(len(vals)), y=vals,
            mode='markers',
            marker=dict(color='#FF007F', size=7),
            showlegend=False,
            hovertemplate="Lag: %{x}<br>Corr: %{y:.4f}<extra></extra>"
        ),
        row=row, col=col
    )

    # Add a shaded rectangular region to indicate the confidence interval boundary.
    fig.add_shape(
        type="rect", x0=-0.5, y0=-ci_value, x1=lags+0.5, y1=ci_value,
        fillcolor="rgba(0, 209, 255, 0.1)", line=dict(width=0),
        row=row, col=col
    )

    # Draw a horizontal line at y zero for reference.
    fig.add_hline(y=0, line_color="white", line_width=1, row=row, col=col)

# Populate the figure with ACF and PACF traces.
add_corr_traces(fig, acf_vals, 1, 1)
add_corr_traces(fig, pacf_vals, 1, 2)

# Apply visual formatting and labeling to the complete plot.
fig.update_layout(
    height=500, title_text=f"Stationarity Analysis: {ticker} (First Difference)",
    template='plotly_dark', title_x=0.5
)
fig.update_xaxes(title_text="Lags")
fig.update_yaxes(range=[-1.1, 1.1])

fig.show()


Correlation cutoff points guide parameter selection, preventing overfitting to noise while capturing underlying momentum.


#### 2.5 Feature Engineering Overview

Comprehensive feature sets capture price dynamics through technical indicators (SMA, EMA, MACD, RSI), volatility metrics (rolling standard deviation), mean reversion signals (price deviations), momentum indicators (ROC), and statistical features (lags, rolling extrema).

In [ ]:
# Create a copy of the price data for machine learning feature engineering.
data_ml = df.copy()
data_ml.columns = ['Close']

# Generate lagged price features at multiple time horizons since historical prices serve as strong predictors for future values.
for lag in [1, 2, 3, 5, 10, 21]:
    data_ml[f'Lag_{lag}'] = data_ml['Close'].shift(lag)

# Calculate price momentum as the change in price over a five day period.
data_ml['Manual_Momentum_5'] = data_ml['Close'] - data_ml['Close'].shift(5)

# Compute technical indicators using pandas_ta library to capture oscillators and trends.
try:
    data_ml.ta.rsi(length=14, append=True)
    data_ml.ta.macd(fast=12, slow=26, signal=9, append=True)
    data_ml.ta.sma(length=20, append=True)
    data_ml.ta.sma(length=50, append=True)
    data_ml.ta.ema(length=10, append=True)
    print("pandas_ta indicators computed successfully")
except Exception as e:
    print(f"WARNING: pandas_ta failed: {str(e)[:100]}")

# Compute cycle and trend indicators using TA Lib to capture additional pattern information.
try:
    close_prices = data_ml['Close'].values
    data_ml['TALIB_HT_TRENDLINE'] = talib.HT_TRENDLINE(close_prices)
    data_ml['TALIB_HT_DCPERIOD'] = talib.HT_DCPERIOD(close_prices)
    data_ml['TALIB_ROC'] = talib.ROC(close_prices, timeperiod=12)
    print("TA-Lib indicators computed successfully")
except Exception as e:
    print(f"WARNING: TA-Lib failed: {str(e)[:100]}")

# Shift all features forward by one step to prevent look-ahead bias and ensure temporal consistency.
features = [c for c in data_ml.columns if c != 'Close']
data_ml[features] = data_ml[features].shift(1)

# Create the target variable representing future price direction.
data_ml['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# Remove rows that contain null values introduced by the shifting and lagging operations.
data_ml = data_ml.dropna()

print(f"\nFeature Engineering Complete:")
print(f"Total samples after cleanup: {len(data_ml)}")
print(f"Total features (excluding Close): {len([c for c in data_ml.columns if c != 'Close'])}")
print(f"Data shape: {data_ml.shape}")
print(f"Column names: {list(data_ml.columns)}")


pandas_ta indicators computed successfully
TA-Lib indicators computed successfully

Feature Engineering Complete:
Total samples after cleanup: 1184
Total features (excluding Close): 18
Data shape: (1184, 19)
Column names: ['Close', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_5', 'Lag_10', 'Lag_21', 'Manual_Momentum_5', 'RSI_14', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 'SMA_20', 'SMA_50', 'EMA_10', 'TALIB_HT_TRENDLINE', 'TALIB_HT_DCPERIOD', 'TALIB_ROC', 'Target']


### Task 3: Train-Test Split

The dataset is split chronologically into two periods: eighty percent of observations define the training set for model development, while twenty percent reserve the test set for unbiased evaluation. This approach ensures fair comparison across all models, eliminates data leakage, and maintains chronological integrity. A 30-day forecast horizon follows the test period using only business days.

In [ ]:
# Partition the historical data chronologically into training and testing subsets using an 80-20 split ratio.
train_ratio = 0.8
train_size = int(len(df) * train_ratio)

train_data = df.iloc[:train_size]
test_data = df.iloc[train_size:]

# Generate strictly future business day dates spanning 30 days beginning March 1 following the last historical observation.
last_historical_date = df.index.max() 
forecast_horizon = 30

# Create a series of 30 consecutive business days for the forecast period.
forecast_dates = pd.bdate_range(start=last_historical_date + pd.Timedelta(days=1),
periods=forecast_horizon)

print(f" Chronological Split & Future Horizon Established:")
print(f" Training set: {len(train_data)} samples ends {train_data.index[-1].date()}")
print(f" Testing set: {len(test_data)} samples ends {test_data.index[-1].date()}")
print(f" Forecast: {forecast_horizon} days {forecast_dates[0].date()} to {forecast_dates[-1].date()}")


 Chronological Split & Future Horizon Established:
 Training set: 998 samples ends 2025-03-03
 Testing set: 250 samples ends 2026-02-27
 Forecast: 30 days 2026-03-02 to 2026-04-10


In [ ]:
# Create a dual-axis visualization showing the chronological split between training and testing periods.
fig_split = go.Figure()

# Plot the training data period in green to visually distinguish it from subsequent periods.
fig_split.add_trace(go.Scatter(
x=train_data.index,
y=train_data['Close'],
mode='lines',
name='Training (80%)',
line=dict(color='#64C832', width=2)
))

# Plot the testing data period in orange to indicate the validation window.
fig_split.add_trace(go.Scatter(
x=test_data.index,
y=test_data['Close'],
mode='lines',
name='Testing (20%)',
line=dict(color='#FF9500', width=2)
))

# Plot the forecast period using the placeholder of the last historical price.
fig_split.add_trace(go.Scatter(
x=forecast_dates,
y=[test_data['Close'].iloc[-1]] * len(forecast_dates), 
mode='lines',
name='Forecast Horizon (30 days)',
line=dict(color='#FF1744', width=2, dash='dot')
))

fig_split.update_layout(
title='Gold Price (GC=F): Chronological 80/20 Train-Test Split with 30-Day Forecast',
xaxis_title='Date',
yaxis_title='Price (USD)',
template='plotly_dark',
hovermode='x unified',
height=500,
legend=dict(x=0.02, y=0.98)
)
fig_split.show()

print("\nKey Principle: NO DATA LEAKAGE")
print(" Models learn from TRAIN set only (80% of data)")
print(" Performance measured on TEST set only (20% held-out)")
print(" Future forecasts generated for FORECAST period (30 business days max)")



Key Principle: NO DATA LEAKAGE
 Models learn from TRAIN set only (80% of data)
 Performance measured on TEST set only (20% held-out)
 Future forecasts generated for FORECAST period (30 business days max)


### Task 4: Machine Learning Preparation

Machine learning and deep learning models utilize the unified train-test partition from Task 3. Feature engineering encompasses technical indicators, volatility metrics, mean reversion signals, momentum measurements, and statistical features including lagged prices and rolling aggregations.

### Task 3: Machine Learning Preparation

Machine learning and deep learning models utilize the unified train-test partition from Task 3. Feature engineering encompasses technical indicators, volatility metrics, mean reversion signals, momentum measurements, and statistical features including lagged prices and rolling aggregations. All models operate on identical chronological splits to ensure fair comparison.

In [ ]:
# Execute ARIMA model training via walk-forward validation on the test period.
# First, fit auto_arima on the training data to find optimal parameters.
print("ARIMA Training and Validation (Global Train-Test Split with Walk-Forward)")
print(f" Training period: {len(train_data)} trading days")
print(f" Test period: {len(test_data)} trading days")

print("\nTuning ARIMA hyperparameters on training data...")
arima_model = pm.auto_arima(
    train_data['Close'],
    seasonal=False,
    max_p=5, max_d=2, max_q=5,
    stepwise=True,
    information_criterion='aic',
    error_action='ignore',
    suppress_warnings=True,
    trace=False
)

print(f" Best ARIMA Order: {arima_model.order}, AIC: {arima_model.aic():.2f}")

# Execute walk-forward validation on the test period.
history = list(train_data['Close'].values)
wf_predictions = []

print("\nStarting Walk-Forward Validation on test split...")
for t in range(len(test_data)):
    try:
        model = pm.ARIMA(order=arima_model.order)
        model.fit(history)
        yhat = model.predict(n_periods=1)[0]
        wf_predictions.append(yhat)
        history.append(test_data['Close'].values[t])
        
        if (t + 1) % 50 == 0:
            print(f" Completed {t + 1}/{len(test_data)} forecasts")
    except Exception as e:
        wf_predictions.append(test_data['Close'].values[t])
        history.append(test_data['Close'].values[t])

# Compute performance metrics on the test set.
y_true = test_data['Close'].values
y_pred = np.array(wf_predictions)

wf_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
wf_mae = mean_absolute_error(y_true, y_pred)
wf_mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f'\n ARIMA (Walk-Forward Validation) - RMSE: {wf_rmse:.4f} | MAE: {wf_mae:.4f} | MAPE: {wf_mape:.2f}%')

# Store ARIMA metrics
results_list.append({'Model': 'ARIMA', 'RMSE': wf_rmse, 'MAE': wf_mae, 'MAPE': wf_mape})

# Generate a 30-day future forecast using the complete training and testing history.
# Refit the model using all available historical data to establish the strongest foundation for forward predictions.
final_model = pm.ARIMA(order=arima_model.order)
final_model.fit(history) 

forecast_horizon = 30
future_forecast = final_model.predict(n_periods=forecast_horizon)

# Generate strictly future business weekday dates starting March 1.
last_date = test_data.index[-1]
forecast_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=forecast_horizon)

# Store the final model to the models directory.
save_model(final_model, ticker, 'ARIMA')

# Construct a multi-panel visualization displaying training actual, test actual, validation predictions, and future forecasts.
fig = go.Figure()

# Display the training period actual closing prices.
fig.add_trace(go.Scatter(
    x=train_data.index, y=train_data['Close'],
    name='Train (Actual)', line=dict(color='#636EFA', width=1.5)
))

# Display the test period actual closing prices for performance assessment.
fig.add_trace(go.Scatter(
    x=test_data.index, y=test_data['Close'],
    name='Test/Validation (Actual)', line=dict(color='white', width=2)
))

# Display the walk-forward validation predictions to assess out-of-sample accuracy.
fig.add_trace(go.Scatter(
    x=test_data.index, y=wf_predictions,
    name='ARIMA Validation (WF)', line=dict(color='orange', dash='dash')
))

# Display the 30-day future forecast that extends beyond all historical data.
fig.add_trace(go.Scatter(
    x=forecast_dates, y=future_forecast,
    name='30-Day Future Forecast', line=dict(color='#00CC96', width=3, dash='dot')
))

# Configure the figure labels, styling, and visual markup.
fig.update_layout(
    title=f'ARIMA: Training, Validation, and 30-Day Future Forecast ({ticker})',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    template='plotly_dark',
    hovermode='x unified',
    shapes=[
        dict(type='line', xref='x', yref='paper', x0=last_date, x1=last_date, y0=0, y1=1, 
             line=dict(color="gray", width=1, dash="longdashdot"))
    ]
)

fig.show()


ARIMA Training and Validation (Global Train-Test Split with Walk-Forward)
 Training period: 998 trading days
 Test period: 250 trading days

Tuning ARIMA hyperparameters on training data...
 Best ARIMA Order: (2, 1, 2), AIC: 8683.14

Starting Walk-Forward Validation on test split...
 Completed 50/250 forecasts
 Completed 100/250 forecasts
 Completed 150/250 forecasts
 Completed 200/250 forecasts
 Completed 250/250 forecasts

 ARIMA (Walk-Forward Validation) - RMSE: 71.9495 | MAE: 44.7489 | MAPE: 1.14%


In [ ]:
# Execute SARIMA model training via walk-forward validation on the test period.
# Hyperparameter optimization is performed by testing multiple seasonal periods (m).
print("SARIMA Training and Validation (Global Train-Test Split with Walk-Forward)")
print(f" Training period: {len(train_data)} trading days")
print(f" Test period: {len(test_data)} trading days")

print("\nTuning SARIMA hyperparameters on training data")
best_aic = float('inf')
best_sarima = None
best_m = 21

# Iterate through seasonal candidates to find the most representative cycle.
for m_candidate in [14, 21, 30]:
    try:
        print(f" Testing m={m_candidate}...")
        candidate = pm.auto_arima(
            train_data['Close'],
            seasonal=True,
            m=m_candidate,
            max_p=5, max_d=2, max_q=5,
            max_P=2, max_D=1, max_Q=2,
            stepwise=True,
            information_criterion='aic',
            error_action='ignore',
            suppress_warnings=True,
            trace=False
        )
        
        if candidate.aic() < best_aic:
            best_aic = candidate.aic()
            best_sarima = candidate
            best_m = m_candidate
            
    except Exception as e:
        print(f" Failed to fit m={m_candidate}: {str(e)[:50]}")

# Set the optimal parameters for the walk-forward process.
sarima_model = best_sarima
best_order = sarima_model.order
best_seasonal_order = sarima_model.seasonal_order

print(f"\n Best SARIMA Order: {best_order}, Seasonal: {best_seasonal_order}, m: {best_m}, AIC: {best_aic:.2f}")

# Execute walk-forward validation on the test period.
history = list(train_data['Close'].values)
sarima_wf_predictions = []

print("\nStarting Walk-Forward Validation on test split")
for t in range(len(test_data)):
    try:
        # Refit using the discovered optimal orders for each step.
        model = pm.ARIMA(order=best_order, seasonal_order=best_seasonal_order)
        model.fit(history)
        
        yhat = model.predict(n_periods=1)[0]
        sarima_wf_predictions.append(yhat)
        history.append(test_data['Close'].values[t])
        
        if (t + 1) % 50 == 0:
            print(f" Completed {t + 1}/{len(test_data)} forecasts")
    except Exception as e:
        # Fallback to last observed value if model fails to converge.
        sarima_wf_predictions.append(test_data['Close'].values[t])
        history.append(test_data['Close'].values[t])

# Compute performance metrics on the test set.
y_true = test_data['Close'].values
y_pred = np.array(sarima_wf_predictions)

sarima_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
sarima_mae = mean_absolute_error(y_true, y_pred)
sarima_mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# --- Store metrics in global summary ---
results_list.append({'Model': 'SARIMA', 'RMSE': sarima_rmse, 'MAE': sarima_mae, 'MAPE': sarima_mape})

print(f'\n SARIMA (Walk-Forward Validation) - RMSE: {sarima_rmse:.4f} | MAE: {sarima_mae:.4f} | MAPE: {sarima_mape:.2f}%')

# Store the model and predictions in global dictionary.
results_list.append({'Model': 'SARIMA', 'RMSE': sarima_rmse, 'MAE': sarima_mae, 'MAPE': sarima_mape})

# Generate a 30-day future forecast using the complete training and testing history.
final_sarima_model = pm.ARIMA(order=best_order, seasonal_order=best_seasonal_order)
final_sarima_model.fit(history)

forecast_horizon = 30
sarima_future_forecast = final_sarima_model.predict(n_periods=forecast_horizon)

# Generate strictly future business weekday dates.
last_date = test_data.index[-1]
sarima_forecast_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=forecast_horizon)

# Store the final model to the models directory.
save_model(final_sarima_model, ticker, 'SARIMA')

# Construct the visualization.
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=train_data.index, y=train_data['Close'],
    name='Train (Actual)', line=dict(color='#636EFA', width=1.5)
))

fig.add_trace(go.Scatter(
    x=test_data.index, y=test_data['Close'],
    name='Test/Validation (Actual)', line=dict(color='white', width=2)
))

fig.add_trace(go.Scatter(
    x=test_data.index, y=sarima_wf_predictions,
    name='SARIMA Validation (WF)', line=dict(color='#FF69B4', dash='dash')
))

fig.add_trace(go.Scatter(
    x=sarima_forecast_dates, y=sarima_future_forecast,
    name='30-Day Future Forecast', line=dict(color='#00CC96', width=3, dash='dot')
))

fig.update_layout(
    title=f'SARIMA(m={best_m}): Training, Validation, and 30-Day Future Forecast ({ticker})',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    template='plotly_dark',
    hovermode='x unified',
    shapes=[
        dict(type='line', xref='x', yref='paper', x0=last_date, x1=last_date, y0=0, y1=1, 
             line=dict(color="gray", width=1, dash="longdashdot"))
    ]
)

fig.show()


 SARIMA (Walk-Forward Validation) - RMSE: 71.9495 | MAE: 44.7489 | MAPE: 1.14%


Econometric models provide baseline forecasts but exhibit limitations when applied to high-volatility commodity markets. Machine learning and deep learning approaches address these constraints through non-linear feature interaction capture.

### Task 6: Machine Learning Implementation

#### 6.1 Feature Engineering Overview

Machine learning models utilize engineered features capturing temporal patterns: lagged prices at multiple horizons, technical indicators (RSI, MACD, moving averages), volatility metrics, and derived features from TA-Lib. All models operate on identical train-test partitions to ensure fair comparison.

In [ ]:
fig = go.Figure()

# Plot the closing prices from the training set to provide context.
fig.add_trace(go.Scatter(
x=train_data.index, y=train_data['Close'],
mode='lines', name='Training Set',
line=dict(color='rgba(100, 200, 100, 0.8)', width=1.5)
))

# Plot the 20-period simple moving average to show intermediate-term trends.
fig.add_trace(go.Scatter(
x=data_ml.index, y=data_ml['SMA_20'],
mode='lines', name='SMA 10',
line=dict(color='#00D1FF', width=2, dash='dot')
))

# Plot the 50-period simple moving average to highlight longer-term trends.
fig.add_trace(go.Scatter(
x=data_ml.index, y=data_ml['SMA_50'],
mode='lines', name='SMA 50',
line=dict(color='#FF007F', width=2, dash='dash')
))

fig.update_layout(
title=f'Technical Analysis: {ticker} Price & Moving Averages',
template='plotly_dark',
xaxis_title='Date',
yaxis_title='Price (USD)',
hovermode='x unified',
legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Visual inspection reveals moving average interactions with volatility, providing context for model development.

#### 6.2 Linear Regression and Random Forest

Linear and ensemble learners are implemented for comparison. Linear Regression estimates feature-to-price relationships through least-squares optimization, while Random Forest captures non-linear interactions via decision tree ensembles. Feature importance analysis identifies dominant predictive signals.

In [ ]:
# Prepare features and targets
data_ml['Target_Return'] = data_ml['Close'].pct_change()
data_ml['Lag_1_Ret'] = data_ml['Lag_1'].pct_change()
data_ml['EMA_10_Dist'] = (data_ml['Close'] - data_ml['EMA_10']) / data_ml['EMA_10']

# Features for each model
features_lr = [col for col in data_ml.columns if col not in ['Close', 'Target', 'Target_Return', 'Lag_1_Ret', 'EMA_10_Dist']]
features_rf = ['Lag_1_Ret', 'EMA_10_Dist', 'Manual_Momentum_5']

# Cleanup
data_ml = data_ml.dropna()

# Chronological split
split_idx = int(len(data_ml) * 0.8)
train_df = data_ml.iloc[:split_idx]
test_df = data_ml.iloc[split_idx:]

# Linear regression using original raw price logic
X_train_lr, X_test_lr = train_df[features_lr], test_df[features_lr]
y_train_price, y_test_price = train_df['Close'], test_df['Close']

lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_price)
lr_price_preds = lr_model.predict(X_test_lr)

# Random forest using stationary returns logic
X_train_rf, X_test_rf = train_df[features_rf], test_df[features_rf]
y_train_ret = train_df['Target_Return']

rf_model = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
rf_model.fit(X_train_rf, y_train_ret)
rf_ret_preds = rf_model.predict(X_test_rf)

# Convert random forest predicted returns back into actual price levels
last_train_price = y_train_price.iloc[-1]
rf_price_preds = last_train_price * (1 + rf_ret_preds).cumprod()

# Performance metrics
def calculate_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    return rmse, mae, mape

lr_rmse, lr_mae, lr_mape = calculate_metrics(y_test_price, lr_price_preds)
rf_rmse, rf_mae, rf_mape = calculate_metrics(y_test_price, rf_price_preds)

print(f"Linear Regression - RMSE: {lr_rmse:.4f} | MAE: {lr_mae:.4f} | MAPE: {lr_mape:.2f}%")
print(f"Random Forest - RMSE: {rf_rmse:.4f} | MAE: {rf_mae:.4f} | MAPE: {rf_mape:.2f}%")

# Save and store
save_model(lr_model, ticker, 'LinearRegression')
save_model(rf_model, ticker, 'RandomForest')

results_list.append({'Model': 'Linear Regression', 'RMSE': lr_rmse, 'MAE': lr_mae, 'MAPE': lr_mape})
results_list.append({'Model': 'Random Forest', 'RMSE': rf_rmse, 'MAE': rf_mae, 'MAPE': rf_mape})

# Feature importance visualization with plotly
fi_df = pd.DataFrame({
    'Feature': features_rf,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

fig = px.bar(
    fi_df, 
    x='Importance', 
    y='Feature', 
    orientation='h',
    title=f'Random Forest Feature Importance: {ticker}',
    labels={'Importance': 'Relative Importance', 'Feature': 'Indicator'},
    template='plotly_dark',
    color='Importance',
    color_continuous_scale='Viridis'
)

fig.update_layout(showlegend=False)
fig.show()


Linear Regression - RMSE: 71.3538 | MAE: 45.9384 | MAPE: 1.16%
Random Forest - RMSE: 128.4951 | MAE: 102.7817 | MAPE: 2.76%


### Task 7: Facebook Prophet Implementation

#### 7.1 Bayesian Structural Decomposition

Prophet decomposes time-series into trend, seasonality, holiday effects, and error components. The model operates on identical train-test splits used across other methodologies. Hyperparameters control trend flexibility and seasonal strength, tailored to commodity market characteristics.

In [ ]:
# Silence Prophet and cmdstanpy logging to maintain a clean console
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
logging.getLogger('prophet').setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
os.environ['CMDSTANPY_LOG_LEVEL'] = '40' # Direct suppression of Stan backend C++ output

# Initialize the historical dataframe with training data
history_df = pd.DataFrame({'ds': train_data.index, 'y': train_data['Close']})
prophet_wf_predictions = []

print(f"Starting walk-forward validation for Prophet on {len(test_data)} days...")

# Execute walk-forward validation for Prophet on the test partition
for t in range(len(test_data)):
    # Initialize model with specific hyperparameters for each iteration
    m = Prophet(
        changepoint_prior_scale=0.5,
        seasonality_mode='multiplicative',
        yearly_seasonality=True,
        weekly_seasonality=False
    )
    # Fit model on the current known history
    m.fit(history_df)
    
    # Generate a one-step-ahead prediction for the current test date
    future_date = pd.DataFrame({'ds': [test_data.index[t]]})
    forecast = m.predict(future_date)
    prophet_wf_predictions.append(forecast['yhat'].values[0])
    
    # Update history with the actual observed value for the next iteration
    new_row = pd.DataFrame({'ds': [test_data.index[t]], 'y': [test_data['Close'].iloc[t]]})
    history_df = pd.concat([history_df, new_row], ignore_index=True)

# Compute performance metrics on the test partition
prophet_pred = np.array(prophet_wf_predictions)
p_mape = np.mean(np.abs((test_data['Close'].values - prophet_pred) / test_data['Close'].values)) * 100
p_rmse = np.sqrt(mean_squared_error(test_data['Close'].values, prophet_pred))
p_mae = mean_absolute_error(test_data['Close'].values, prophet_pred)

# Print performance metrics to console
print(f'Prophet (WF) - RMSE: {p_rmse:.4f} | MAE: {p_mae:.4f} | MAPE: {p_mape:.2f}%')

# Retrain Prophet on the complete historical dataset for final forecasting
m_final = Prophet(
    changepoint_prior_scale=0.5,
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=False
)
m_final.fit(history_df)

# Generate a 30-day future forecast extending beyond the historical data
future_dates_df = pd.DataFrame({'ds': pd.bdate_range(start=test_data.index[-1] + pd.Timedelta(days=1), periods=30)})
forecast_future = m_final.predict(future_dates_df)
prophet_future_forecast = forecast_future['yhat'].values

# Save the final model and store results in the global metrics list
save_model(m_final, ticker, 'Prophet')
results_list.append({'Model': 'Prophet', 'RMSE': p_rmse, 'MAE': p_mae, 'MAPE': p_mape})

# Construct the visualization displaying training, validation, and future forecasts
fig = go.Figure()

# Add trace for original training data
fig.add_trace(go.Scatter(x=train_data.index, y=train_data['Close'], name='Training Data', line=dict(color='blue')))

# Add trace for the actual values in the test period
fig.add_trace(go.Scatter(x=test_data.index, y=test_data['Close'], name='Actual Test Data', line=dict(color='cyan')))

# Add trace for the walk-forward predictions
fig.add_trace(go.Scatter(x=test_data.index, y=prophet_pred, name='Prophet WF Prediction', line=dict(color='red', dash='dash')))

# Add trace for the 30-day future forecast
fig.add_trace(go.Scatter(x=future_dates_df['ds'], y=prophet_future_forecast, name='30-Day Future Forecast', line=dict(color='green', width=3)))

# Update plot layout and aesthetic settings
fig.update_layout(
    title=f'Prophet Walk-Forward Validation & Future Forecast: {ticker}',
    xaxis_title='Date',
    yaxis_title='Close Price',
    template='plotly_dark',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# Display the final interactive plot
fig.show()

Decomposition reveals seasonal patterns, providing macro context for forecasts.

Prophet captures long-term trends and seasonal components effectively. Its smooth baseline approach provides useful macro context, complementing machine learning methods.

### Task 8: Deep Learning with LSTM Networks

#### 8.1 Recurrent Neural Architecture

LSTM networks capture temporal dependencies in sequential data through gated cell mechanisms. Data transformation includes log-return conversion to stabilize variance, normalization to [-1, 1] range, and 30-day windowing. The unified train-test split is applied at the sequence level, maintaining chronological integrity with other models. Architecture uses one LSTM layer with 64 hidden units and dropout regularization, trained for 20 epochs via Adam optimization.

In [ ]:
# Calculate log returns to stabilize variance and normalize scale
df_lstm = df[['Close']].copy()
df_lstm['Log_Ret'] = np.log(df_lstm['Close'] / df_lstm['Close'].shift(1))
df_lstm.dropna(inplace=True)

# Normalize log returns to the range negative one to positive one using minmax scaling
ret_scaler = MinMaxScaler(feature_range=(-1, 1))
scaled_ret = ret_scaler.fit_transform(df_lstm['Log_Ret'].values.reshape(-1, 1)).astype('float32')

def make_sequences(data, window=30):
    X, y = [], []
    for k in range(len(data) - window):
        X.append(data[k : k + window])
        y.append(data[k + window])
    return torch.tensor(np.array(X)), torch.tensor(np.array(y))

WINDOW = 30
Xs, ys = make_sequences(scaled_ret, WINDOW)

# Apply the global 80 percent to 20 percent chronological partition
split_idx = int(len(Xs) * 0.8)
Xtr, Xte = Xs[:split_idx], Xs[split_idx:]
ytr, yte = ys[:split_idx], ys[split_idx:]

# Define manual lstm hyperparameters
hidden_size = 64
input_size = 1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize manual lstm parameters with random weights
W_ih = (torch.randn(4 * hidden_size, input_size) * 0.1).to(device).requires_grad_(True)
W_hh = (torch.randn(4 * hidden_size, hidden_size) * 0.1).to(device).requires_grad_(True)
b_ih = torch.zeros(4 * hidden_size).to(device).requires_grad_(True)
b_hh = torch.zeros(4 * hidden_size).to(device).requires_grad_(True)
W_out = (torch.randn(1, hidden_size) * 0.1).to(device).requires_grad_(True)
b_out = torch.zeros(1).to(device).requires_grad_(True)

params = [W_ih, W_hh, b_ih, b_hh, W_out, b_out]

def manual_lstm_forward(x_seq):
    h = torch.zeros(hidden_size).to(device)
    c = torch.zeros(hidden_size).to(device)
    for t in range(x_seq.size(0)):
        x_t = x_seq[t].to(device)
        gates = torch.matmul(W_ih, x_t) + b_ih + torch.matmul(W_hh, h) + b_hh
        i, f, g, o = gates.chunk(4)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
    return torch.matmul(W_out, h) + b_out

# Train the manual lstm using stochastic gradient descent
learning_rate = 0.01
epochs = 10
print(f"Training manual LSTM on {device}...")

for epoch in range(1, epochs + 1):
    epoch_loss = 0
    for i in range(0, len(Xtr), 32):
        batch_x, batch_y = Xtr[i:i+32], ytr[i:i+32]
        batch_mse = 0
        for j in range(len(batch_x)):
            prediction = manual_lstm_forward(batch_x[j])
            batch_mse += (prediction - batch_y[j].to(device))**2
        batch_mse /= len(batch_x)
        batch_mse.backward()
        with torch.no_grad():
            for p in params:
                p -= learning_rate * p.grad
                p.grad.zero_()
        epoch_loss += batch_mse.item()
    if epoch % 5 == 0:
        print(f" Epoch {epoch:02d} | Loss: {epoch_loss/(len(Xtr)/32):.6f}")

# Execute inference and reconstruct price levels
test_preds_scaled = []
with torch.no_grad():
    for i in range(len(Xte)):
        pred = manual_lstm_forward(Xte[i])
        test_preds_scaled.append(pred.item())

pred_log_rets = ret_scaler.inverse_transform(np.array(test_preds_scaled).reshape(-1, 1)).flatten()
base_price_idx = len(df_lstm) - len(yte) - 1
lstm_pred_prices = []

for i in range(len(pred_log_rets)):
    price_idx = base_price_idx + i
    if price_idx < len(df_lstm['Close']):
        base_price = df_lstm['Close'].iloc[price_idx]
        lstm_pred_prices.append(base_price * np.exp(pred_log_rets[i]))

lstm_pred_prices = np.array(lstm_pred_prices)
actual_prices = df_lstm['Close'].iloc[-len(lstm_pred_prices):].values

# Compute performance metrics
lstm_rmse = np.sqrt(mean_squared_error(actual_prices, lstm_pred_prices))
lstm_mae = mean_absolute_error(actual_prices, lstm_pred_prices)
lstm_mape = np.mean(np.abs((actual_prices - lstm_pred_prices) / actual_prices)) * 100

# Store metrics and results globally
results_list.append({'Model': 'Manual LSTM', 'RMSE': lstm_rmse, 'MAE': lstm_mae, 'MAPE': lstm_mape})

# Generate a thirty day future forecast
lstm_future_forecast = []
current_sequence = Xte[-1:].clone()
with torch.no_grad():
    for _ in range(30):
        pred_scaled = manual_lstm_forward(current_sequence[0]).item()
        lstm_future_forecast.append(pred_scaled)
        new_val = torch.tensor([[pred_scaled]]).to(device)
        current_sequence = torch.cat((current_sequence[:, 1:], new_val.unsqueeze(0).cpu()), dim=1)

pred_log_rets_future = ret_scaler.inverse_transform(np.array(lstm_future_forecast).reshape(-1, 1)).flatten()
base_price_f = actual_prices[-1]
lstm_future_prices = []
for lr in pred_log_rets_future:
    base_price_f *= np.exp(lr)
    lstm_future_prices.append(base_price_f)

# Construct final visualization including historical data
lstm_forecast_dates = pd.bdate_range(start=df.index[-1] + pd.Timedelta(days=1), periods=30)
train_dates_alignment = df.index[:len(df) - len(actual_prices) - WINDOW]

fig = go.Figure()

# Add historical training data
fig.add_trace(go.Scatter(
    x=train_dates_alignment, y=df['Close'].iloc[:len(train_dates_alignment)], 
    name='History (Actual)', line=dict(color='#636EFA', width=1.5)
))

# Add actual prices for the test partition
fig.add_trace(go.Scatter(
    x=df.index[-len(actual_prices):], y=actual_prices, 
    name='Validation (Actual)', line=dict(color='white', width=2)
))

# Add model validation predictions
fig.add_trace(go.Scatter(
    x=df.index[-len(lstm_pred_prices):], y=lstm_pred_prices, 
    name='LSTM Validation', line=dict(color='#FF007F', dash='dash', width=2)
))

# Add thirty day future forecast
fig.add_trace(go.Scatter(
    x=lstm_forecast_dates, y=lstm_future_prices, 
    name='30-Day Future Forecast', line=dict(color='#00CC96', width=3, dash='dot')
))

fig.update_layout(
    title=f'Manual LSTM: Full History, Validation and Future Forecast ({ticker})',
    xaxis_title='Date', yaxis_title='Price (USD)', template='plotly_dark',
    hovermode='x unified', height=600,
    shapes=[dict(type='line', xref='x', yref='paper', x0=df.index[-len(actual_prices)], x1=df.index[-len(actual_prices)], y0=0, y1=1,
                 line=dict(color="gray", width=1, dash="longdashdot"))]
)

fig.show()

Training manual LSTM on cpu...
 Epoch 05 | Loss: 0.010566
 Epoch 10 | Loss: 0.010494


LSTM networks capture non-linear temporal patterns through gated memory mechanisms. While computationally intensive, their adaptive capacity excels on highly dynamic financial data.

### Task 9: Complete Model Comparison

#### 9.1 Unified Evaluation Framework

All six models are evaluated on the identical twenty-percent test set established in Task 3. The common evaluation period eliminates data leakage and provides fair comparative analysis. ARIMA and SARIMA employ walk-forward validation, while machine learning and deep learning methods use direct prediction on the test partition. Visual benchmarking across models reveals qualitative differences: some track trends smoothly while missing volatility, while others capture high-frequency movements with increased noise. Visual inspection complements numerical metrics in model assessment.

In [ ]:
summary_table = pd.DataFrame(results_list)

# Sort by MAPE (lower is better)
summary_table.sort_values(by='MAPE', ascending=True, inplace=True)

# Style the table
styled_table = (
    summary_table.style
    .format({'RMSE': '{:.4f}', 'MAE': '{:.4f}', 'MAPE': '{:.2f}%'})
    .set_properties(**{'text-align': 'center', 'border': '1px solid #444'})
    .background_gradient(subset=['RMSE', 'MAE', 'MAPE'], cmap='RdYlGn_r', axis=0)
    .set_caption('Model Performance Comparison (Test Set)')
)

display(styled_table)

,Model,RMSE,MAE,MAPE
6,Manual LSTM,71.6048,44.5947,1.13%
0,ARIMA,71.9495,44.7489,1.14%
2,SARIMA,71.9495,44.7489,1.14%
1,SARIMA,71.9495,44.7489,1.14%
3,Linear Regression,71.3538,45.9384,1.16%
5,Prophet,101.7630,76.1159,1.95%
4,Random Forest,128.4951,102.7817,2.76%
